In [10]:
%cd /content/aegis-guard

from pathlib import Path

Path("src/model").mkdir(parents=True, exist_ok=True)
Path("src/model/__init__.py").touch()

print("Model infrastructure ready.")

/content/aegis-guard
Model infrastructure ready.


In [11]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

# ---------------------------------------------------------
# Get Hugging Face token from Colab Secrets
# ---------------------------------------------------------
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN was not found in Colab Secrets. "
        "Add a secret named HF_TOKEN and try again."
    )

print("HF token: detected")
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.cuda.is_available())

# ---------------------------------------------------------
# 4-bit quantization
# ---------------------------------------------------------
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

# ---------------------------------------------------------
# Tokenizer
# ---------------------------------------------------------
print("\nLoading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    use_fast=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded.")

# ---------------------------------------------------------
# Model
# ---------------------------------------------------------
print("\nLoading 4-bit model...")
print("This may take several minutes.")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

model.eval()

print("\n" + "=" * 70)
print("MODEL LOAD SUCCESSFUL")
print("=" * 70)

print("Model:", MODEL_ID)
print("Device map:", model.hf_device_map)
print("Dtype:", model.dtype)

if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3

    print(f"GPU memory allocated: {allocated:.2f} GB")
    print(f"GPU memory reserved:  {reserved:.2f} GB")
    print(f"GPU total VRAM:      {total:.2f} GB")

print("=" * 70)

HF token: detected
GPU: Tesla T4
CUDA: True

Loading tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Tokenizer loaded.

Loading 4-bit model...
This may take several minutes.


...

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]


MODEL LOAD SUCCESSFUL
Model: meta-llama/Llama-3.1-8B-Instruct
Device map: {'': 0}
Dtype: torch.float16
GPU memory allocated: 5.31 GB
GPU memory reserved:  6.73 GB
GPU total VRAM:      14.56 GB


In [13]:
prompt = "Explain in simple terms what machine learning is."

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": prompt},
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        inputs,
        max_new_tokens=128,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

new_tokens = outputs[0][inputs.shape[-1]:]

response = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True,
)

print("\nMODEL RESPONSE")
print("-" * 70)
print(response)
print("-" * 70)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



MODEL RESPONSE
----------------------------------------------------------------------
Machine learning is a way for computers to learn from data without being explicitly programmed.

Imagine you have a lot of pictures of dogs and cats, and you want a computer to be able to tell the difference between them. You wouldn't tell the computer "if it has four legs and a tail, it's a dog." Instead, you would show the computer many pictures of dogs and cats, and let it figure out the patterns and characteristics that make them different.

The computer would look at the pictures, notice things like the shape of the ears, the color of the fur, and the way the animal is sitting, and use that information to
----------------------------------------------------------------------
